# 12. Gépi Tanulásos Ármeghatározás és Piaci Arbitrázs

**Kontextus és Kapcsolódás:** Az eddigi parametrikus modellek (OLS, SAR, GWR) jól magyaráznak, de a nemlineáris interakciókat nehezen kezelik. A prediktív pontosság maximalizálása érdekében most áttérünk a gépi tanulásra (Random Forest).

**Cél**: Random Forest árbecslő modell tanítása, a változók fontossági rangsorának (Feature Importance) feltárása és a piacilag alulárazott (arbitrázs) lakások automatikus azonosítása.

---

### 📖 Mi az a piaci arbitrázs?
Ha a gépi tanulási modell nagy pontossággal ($R^2 > 0.80$) megbecsüli a lakás reális értékét, a modell jóslatánál jóval olcsóbban hirdetett ingatlanok **alulárazott arbitrázs lehetőséget** jelentenek.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Modellezésre elérhető eladó lakások: {len(elado)} db.")

### 1. Random Forest Modell Illeszkedés és Diagnosztika

**📌 Mennyire pontos az algoritmus?**
A Random Forest modell az adatok több mint 80%-át magyarázza a teszthalmazon, átlagosan 3-4 millió forintos hibahatárral becsülve meg a vételárat.

In [ ]:
df_pontos['emelet_szam'] = df_pontos['emelet_szam'].fillna(df_pontos['emelet_szam'].median())
df_pontos['epulet_kora_ev'] = df_pontos['epulet_kora_ev'].fillna(df_pontos['epulet_kora_ev'].median())

features = [
    'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel', 
    'has_lift', 'allapot_kod', 'van_erkely', 'emelet_szam', 'epulet_kora_ev',
    'tavolsag_metro_halozati_m', 'tavolsag_vasut_m', 'tavolsag_vasut_halozati_m', 
    'tavolsag_mazsa_halozati_m'
]
df_ml = df_pontos.dropna(subset=['nm_ar_huf', 'price_huf'] + features).copy()

X = df_ml[features]
y = df_ml['nm_ar_huf']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

rf = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

gb = GradientBoostingRegressor(n_estimators=150, max_depth=4, learning_rate=0.08, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mape_rf = np.mean(np.abs((y_test - y_pred_rf) / y_test)) * 100

r2_gb = r2_score(y_test, y_pred_gb)
mae_gb = mean_absolute_error(y_test, y_pred_gb)

kpi_cards = [
    ("Random Forest R²", f"{r2_rf:.3f}", "Teszt adathalmazon", "#1e3a8a"),
    ("Gradient Boosting R²", f"{r2_gb:.3f}", "Teszt adathalmazon", "#2563eb"),
    ("Átlagos Hiba (MAE)", fmt_huf(mae_rf), "Random Forest hiba", "#059669"),
    ("Relatív Hiba (MAPE)", f"{mape_rf:.1f}%", "Százalékos pontosság", "#10b981"),
    ("Tanító Minta (N)", f"{len(X_train)} db", "80% arány", "#d97706"),
    ("Tesztelő Minta (N)", f"{len(X_test)} db", "20% arány", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Változók Relatív Fontossága (Feature Importance MDI)

**📌 Mely tulajdonságok a legfontosabbak?**
1. Korrigált alapterület (m²) – a legfőbb magyarázó erő,
2. Lokáció (metró és vasút távolság),
3. Műszaki állapotindex,
4. Építőanyag (panel vs. tégla).

In [ ]:
# 1. Tényleges vs Becsült ábra
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=y_test / 1e3, y=y_pred_rf / 1e3,
    mode='markers', marker=dict(color='#2563eb', opacity=0.7, size=8),
    name='Teszt adatok'
))
min_p = min(y_test.min(), y_pred_rf.min()) / 1e3
max_p = max(y_test.max(), y_pred_rf.max()) / 1e3
fig1.add_trace(go.Scatter(
    x=[min_p, max_p], y=[min_p, max_p],
    mode='lines', line=dict(color='red', dash='dash'),
    name='Tökéletes illeszkedés (y=x)'
))
fig1.update_layout(
    title='Random Forest: Tényleges vs. Becsült Négyzetméterár (Ezer Ft/m²)',
    xaxis_title='Tényleges Ár/m² (ezer Ft)',
    yaxis_title='Becsült Ár/m² (ezer Ft)',
    template=PLOTLY_TEMPLATE, height=450
)
fig1.show()

# 2. Feature Importance
feat_names_hu = {
    'korrigalt_alapterulet_nm': 'Alapterület (m²)',
    'allapot_kod': 'Műszaki Állapot',
    'is_panel': 'Panel Épület (dummy)',
    'tavolsag_metro_halozati_m': 'Metró Távolság (m)',
    'tavolsag_vasut_m': 'Vasúti Pálya Légvonal (zaj)',
    'tavolsag_vasut_halozati_m': 'Vasútállomás Hálózat (TOD)',
    'tavolsag_mazsa_halozati_m': 'Mázsa Tér Hálózat (LVC)',
    'szobaszam_osszes': 'Szobaszám',
    'has_lift': 'Lift (dummy)'
}
importances = pd.DataFrame({
    'Valtozo': [feat_names_hu.get(f, f) for f in features],
    'Fontossag': rf.feature_importances_
}).sort_values('Fontossag', ascending=True)

fig2 = px.bar(
    importances, x='Fontossag', y='Valtozo', orientation='h',
    title='Ármeghatározó Tényezők Relatív Fontossága (Random Forest Gini Importance)',
    labels={'Fontossag': 'Relatív Fontosság (0 - 1)', 'Valtozo': 'Jellemző'},
    color='Fontossag', color_continuous_scale='Blues',
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=420)
fig2.show()

### 3. A Legjobb Piaci Arbitrázs Lehetőségek Toplistája

**📌 Mit tartalmaz a táblázat?**
A modell által azonosított legnagyobb mértékben alulárazott kőbányai lakások listáját a hirdetett és becsült ár különbségével.

In [ ]:
# Teljes minta előrejelzése a legjobb modellel (Random Forest)
df_ml['becsult_nm_ar'] = rf.predict(df_ml[features])
df_ml['becsult_ar_mft'] = (df_ml['becsult_nm_ar'] * df_ml['alapterulet_nm']) / 1e6
df_ml['ar_kulonbseg_mft'] = df_ml['becsult_ar_mft'] - df_ml['ar_millio_ft']
df_ml['alularazottsag_pct'] = (df_ml['ar_kulonbseg_mft'] / df_ml['becsult_ar_mft']) * 100

# Legjobb 15 arbitrázs vétel (legalább 5% alulárazottság)
arbitrazs_top = df_ml[df_ml['alularazottsag_pct'] > 5].sort_values('alularazottsag_pct', ascending=False).head(15).copy()

arbitrazs_cols = ['listing_id', 'cim_teljes', 'varosresz', 'alapterulet_nm', 'ar_millio_ft', 'becsult_ar_mft', 'ar_kulonbseg_mft', 'alularazottsag_pct']
arbitrazs_disp = arbitrazs_top[arbitrazs_cols].copy()
arbitrazs_disp.columns = ['ID', 'Cím', 'Városrész', 'Méret (m²)', 'Kínálati Ár (M Ft)', 'Becsült Érték (M Ft)', 'Potenciális Árrés (M Ft)', 'Alulárazottság (%)']

html_arb = "<div style='overflow-x:auto; margin: 15px 0;'>" + arbitrazs_disp.round(1).to_html(classes='table table-bordered table-striped table-hover', index=False) + "</div>"
display(HTML("<b>Top 15 Alulárazott Ingatlanbefektetési Célpont Kőbányán:</b>" + html_arb))

# Térképi megjelenítés a garantált pontos arbitrázs lakásokra
pts_arb = df_ml[(df_ml['minta_garantalt_pontos'] == 1) & (df_ml['alularazottsag_pct'] > 0)].copy()
if len(pts_arb) > 0:
    fig3 = px.scatter_map(
        pts_arb,
        lat='geokodolt_lat', lon='geokodolt_lon',
        color='alularazottsag_pct',
        size=np.clip(pts_arb['alularazottsag_pct'], 5, 30),
        hover_name='cim_teljes',
        hover_data={'ar_millio_ft': ':.1f', 'becsult_ar_mft': ':.1f', 'alularazottsag_pct': ':.1f'},
        color_continuous_scale='Viridis',
        zoom=12.2,
        center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
        map_style='carto-positron',
        title='Alulárazott Ingatlanok Térképi Elhelyezkedése (Alulárazottság mértéke szerint)'
    )
    fig3.update_layout(height=480, margin={"r":0,"t":40,"l":0,"b":0})
    fig3.show()

### 4. Interaktív Árbecslő Kalkulátor

**📌 Próbálja ki az árbecslőt!**
Adja meg a lakás méretét, állapotát, szobaszámát és elhelyezkedését, és a Random Forest modell azonnal megbecsüli a piaci értéket!

In [ ]:
w_terulet = widgets.IntSlider(min=25, max=120, value=55, description='Méret (m²):')
w_allapot = widgets.Dropdown(options=[('Felújított (5)', 5), ('Jó állapotú (4)', 4), ('Közepes (3)', 3), ('Felújítandó (2)', 2), ('Új építésű (6)', 6)], value=4, description='Állapot:')
w_panel = widgets.RadioButtons(options=[('Tégla', 0), ('Panel', 1)], value=0, description='Típus:')
w_metro_dist = widgets.IntSlider(min=100, max=2500, step=100, value=600, description='Metró (m):')
w_vasut_dist = widgets.IntSlider(min=50, max=1500, step=50, value=400, description='Vasút (m):')

out_calc = widgets.Output()

def szamol_ertek(*args):
    x_input = pd.DataFrame([{
        'korrigalt_alapterulet_nm': float(w_terulet.value),
        'szobaszam_osszes': 2.0 if w_terulet.value < 60 else 3.0,
        'is_panel': float(w_panel.value),
        'has_lift': 1.0 if w_panel.value == 1 else 0.0,
        'allapot_kod': float(w_allapot.value),
        'tavolsag_metro_halozati_m': float(w_metro_dist.value),
        'tavolsag_vasut_m': float(w_vasut_dist.value),
        'tavolsag_vasut_halozati_m': float(w_vasut_dist.value * 1.2),
        'tavolsag_mazsa_halozati_m': 1000.0
    }])
    pred_nm = rf.predict(x_input)[0]
    pred_tot = (pred_nm * w_terulet.value) / 1e6
    
    with out_calc:
        clear_output(wait=True)
        pred_kpis = [
            ("Becsült Lakásár", fmt_mft(pred_tot), "ML modell előrejelzés", "#10b981"),
            ("Becsült Fajlagos Ár", fmt_huf(pred_nm), "Kínálati becslés", "#2563eb"),
            ("Modell Típus", "Random Forest Regressor", "150 döntési fa", "#7c3aed")
        ]
        display(HTML(kpi_grid_html(pred_kpis)))

w_terulet.observe(szamol_ertek, names='value')
w_allapot.observe(szamol_ertek, names='value')
w_panel.observe(szamol_ertek, names='value')
w_metro_dist.observe(szamol_ertek, names='value')
w_vasut_dist.observe(szamol_ertek, names='value')

display(widgets.VBox([
    widgets.HBox([w_terulet, w_allapot]),
    widgets.HBox([w_panel, w_metro_dist, w_vasut_dist])
]))
display(out_calc)
szamol_ertek()